# Stage 2 — Validation

(i) DCM null (50 draws), (ii) seeded Louvain/Infomap comparison, (iii) five-schedule reseed campaign with matched-control ablations and the dark-matter stream, (iv) self-containment via the consensus table; then the arXiv label check. Long stages (hours) sit behind `RUN_LONG`; their products of record ship in `data/communities/` and `data/communities/campaign/` (the campaign ledger).

In [1]:
import os, pathlib, sys
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "scripts").is_dir() and (root / "data").is_dir(): break
    root = root.parent
else:
    raise SystemExit("repository root (containing scripts/ and data/) not found within 6 levels")
os.chdir(root); print("working directory:", os.getcwd())
assert "igraph" in {m.split("==")[0] for m in os.popen(f"{sys.executable} -m pip list --format=freeze 2>/dev/null").read().split()}, \
    f"kernel {sys.executable} lacks python-igraph: select the grb-venv kernel (see notebooks/README.md)"

working directory: /Users/salim/Desktop/Projects/Astrograph/GRB_Community_Structure


In [2]:
CORPUS = "data/raw/ads_corpus_v2_core_frozen.jsonl"  # local-only frozen corpus (ADS terms); see README
import pathlib
HAVE_CORPUS = pathlib.Path(CORPUS).exists()
print("frozen corpus present:", HAVE_CORPUS)
RUN_LONG = False
RUN_FETCH = False

frozen corpus present: True


(i) DCM null: fifty draws at ΔT = 1 yr and the ΔT = 2 yr check (long; `--realisations 50` is required — the script default is 20).

In [3]:
if RUN_LONG and HAVE_CORPUS:
    %run scripts/dcm_null.py --layer-years 1 --realisations 50
    %run scripts/dcm_null.py --layer-years 2 --realisations 50
else:
    print("skipped; products of record: dcm_null_dT1.json / dcm_null_dT2.json")

skipped; products of record: dcm_null_dT1.json / dcm_null_dT2.json


(ii) Cross-algorithm agreement on the frozen graph, seeded.

In [4]:
if HAVE_CORPUS:
    %run scripts/algorithm_comparison2.py
else:
    print("skipped; product of record: algorithm_comparison2.json")

graph: 13,800 nodes, 378,832 weighted edges


  louvain              ARI 0.661  NMI 0.721  one-to-one 0.767  nesting purity 0.808  12 communities >=30


  infomap              ARI 0.579  NMI 0.688  one-to-one 0.643  nesting purity 0.889  43 communities >=30


  leading_eigenvector  ARI 0.112  NMI 0.280  one-to-one 0.358  nesting purity 0.358  4 communities >=30


wrote /Users/salim/Desktop/Projects/Astrograph/GRB_Community_Structure/data/communities/algorithm_comparison2.json


(iii-a) Representation checks.

In [5]:
if HAVE_CORPUS:
    %run scripts/representation_check.py
if RUN_LONG and HAVE_CORPUS:
    %run scripts/representation_three.py
    %run scripts/canonical_directed.py
else:
    print("consensus-level representation products are in data/communities/")

multigraph 13,800 nodes 380,361 edges


collapsed  13,800 nodes 378,832 edges (weights sum 380,361)



fixed-membership equivalence: 0.464804780707 == 0.464804780707  PASS



  seed 42: Q_multi=0.46275  Q_collapsed=0.46269  cross-ARI=0.974


  seed 43: Q_multi=0.46512  Q_collapsed=0.46366  cross-ARI=0.940


...

  seed 48: Q_multi=0.46321  Q_collapsed=0.46448  cross-ARI=0.850


  seed 49: Q_multi=0.46353  Q_collapsed=0.46267  cross-ARI=0.895


  seed 50: Q_multi=0.46329  Q_collapsed=0.46353  cross-ARI=0.762


  seed 51: Q_multi=0.45953  Q_collapsed=0.45877  cross-ARI=0.869



cross-representation ARI, matched seed : 0.875 +/- 0.060  (n=10)
within multigraph, reseed              : 0.648 +/- 0.087
within collapsed,  reseed              : 0.669 +/- 0.083

=> representation moves the partition LESS than reseeding does
wrote /Users/salim/Desktop/Projects/Astrograph/GRB_Community_Structure/data/communities/representation_check.json
consensus-level representation products are in data/communities/


(iii-b) The five-schedule reseed campaign (hours) and the base ablation-control pass that the repository-only pipeline diagram reads; the aggregate is cheap and always runs on the saved ledger.

In [6]:
if RUN_LONG and HAVE_CORPUS:
    for j in range(5):
        %run scripts/campaign_reseed.py --j {j}
    %run scripts/ablation_controls2.py
%run scripts/campaign_aggregate.py

{
 "n_realizations": 5,
 "blocks": [
  42,
  10042,
  20042,
  30042,
  40042
 ],
 "ablations": {
  "no reference list": {
   "delta_ari": {
    "values": [
     0.0443,
     -0.0231,
     -0.0235,
     -0.0239,
     -0.019
    ],
    "median": -0.0231,
    "range": [
     -0.0239,
     0.0443
    ]
   },
   "pct_rank": {
    "values": [
     0.92,
     0.4,
     0.28,
     0.36,
     0.36
    ],
    "median": 0.36,
    "range": [
     0.28,
     0.92
    ]
   },
   "m_tail": 0,
   "n_nonconverged": [
    0,
    0,
    0,
    0,
    0
   ]
  },
  "not refereed": {
   "delta_ari": {
    "values": [
     -0.0958,
     0.0486,
     -0.0413,
     -0.0466,
     -0.0313
    ],
    "median": -0.0413,
    "range": [
     -0.0958,
     0.0486
    ]
   },
   "pct_rank": {
    "values": [
     0.0,
     0.88,
     0.28,
     0.04,
     0.24
    ],
    "median": 0.24,
    "range": [
     0.0,
     0.88
    ]
   },
   "m_tail": 2,
   "n_nonconverged": [
    0,
    0,
    0,
    0,
    0
   ]
  },
 

arXiv primary-category check on the labels (the metadata file ships with the repository; refetch only to rebuild it).

In [7]:
if RUN_FETCH:
    %run scripts/fetch_arxiv_meta.py
%run scripts/validate_labels_arxiv.py

9,813 core papers carry an arXiv primary category

corpus-wide top categories:
  astro-ph.HE         4,865  49.6%
  astro-ph            3,227  32.9%
  astro-ph.CO           558   5.7%
  astro-ph.IM           262   2.7%
  gr-qc                 235   2.4%
  astro-ph.GA           197   2.0%

per community, categories enriched over their corpus share:

  C0   BATSE-era distance scale             [ 576/2126 27%]
        astro-ph 54% (x1.7) | gr-qc 3% (x1.4) | astro-ph.CO 6% (x1.1)
  C1   afterglows                           [1582/2120 75%]
        astro-ph 59% (x1.8) | astro-ph.IM 2% (x0.9) | astro-ph.HE 35% (x0.7)
  C2   compact mergers, short GRBs          [1787/1999 89%]
        gr-qc 6% (x2.3) | astro-ph.HE 75% (x1.5) | astro-ph.IM 3% (x1.2)
  C3   prompt emission, radiation physics   [1274/1674 76%]
        astro-ph.IM 4% (x1.6) | astro-ph.HE 61% (x1.2) | astro-ph 29% (x0.9)
  C4   high-energy neutrinos                [1079/1459 74%]
        hep-ph 7% (x3.7) | astro-ph.HE 61% (x1.2) | 